<a href="https://colab.research.google.com/github/jonrenzo/Telcovantage-Site-Map-Reader/blob/master/colab/Telcovantage_GPU_OCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telcovantage — GPU TrOCR Server (Cloudflare Tunnel)

Runs TrOCR models on Colab GPU and exposes them via Cloudflare Tunnel.

**Workflow:**
1. Run all cells below
2. Copy the `REMOTE_TROCR_URL` printed at the end
3. On your laptop: `$env:REMOTE_TROCR_URL = "<url>"` then `py server.py`

---
## Cell 1: Install dependencies

In [47]:
!pip install -q fastapi uvicorn

---
## Cell 2: Install Cloudflare Tunnel

In [48]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!cloudflared --version

Selecting previously unselected package cloudflared.
(Reading database ... 122363 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.5.0) ...
Setting up cloudflared (2026.5.0) ...
Processing triggers for man-db (2.10.2-1) ...
cloudflared version 2026.5.0 (built 2026-05-13-11:24 UTC)


---
## Cell 3: Clone your repo

In [49]:
import os
REPO_URL = "https://github.com/jonrenzo/Telcovantage-Site-Map-Reader"
REPO_DIR = "/content/Telcovantage-Site-Map-Reader"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Already cloned at {REPO_DIR}")
    !cd {REPO_DIR} && git pull

Already cloned at /content/Telcovantage-Site-Map-Reader
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 702 bytes | 702.00 KiB/s, done.
From https://github.com/jonrenzo/Telcovantage-Site-Map-Reader
   ab91241..d250c7f  master     -> origin/master
Updating ab91241..d250c7f
Fast-forward
 colab/Telcovantage_GPU_OCR.ipynb | 86 ++++++++++++++++++++--------------------
 1 file changed, 43 insertions(+), 43 deletions(-)


---
## Cell 4: Install project Python dependencies

In [50]:
!pip install -q -r {REPO_DIR}/requirements.txt
print("Dependencies installed.")

Dependencies installed.


---
## Cell 5: Start TrOCR server (non-blocking)

In [51]:
import threading, time, sys, requests
sys.path.insert(0, REPO_DIR)

import uvicorn
from colab.server import app

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

t = threading.Thread(target=run_server, daemon=True)
t.start()

# Wait for server to start
for i in range(30):
    time.sleep(1)
    try:
        r = requests.get("http://localhost:8000/health", timeout=3)
        if r.status_code == 200:
            print("Server started on port 8000.")
            break
    except Exception:
        pass

# Pre-warm the model
print("Loading TrOCR model on GPU (first run may take ~15-30s)...")
resp = requests.post("http://localhost:8000/ocr/pole",
    json={"segments": [{"x1":0,"y1":0,"x2":10,"y2":0},
                      {"x1":5,"y1":0,"x2":5,"y2":20}],
          "bbox": [0,0,10,20], "auto_rotate": False}, timeout=180)
print("Model loaded:", resp.status_code)

INFO:     Started server process [1342]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     127.0.0.1:48512 - "GET /health HTTP/1.1" 200 OK
Server started on port 8000.
Loading TrOCR model on GPU (first run may take ~15-30s)...
INFO:     127.0.0.1:48528 - "POST /ocr/pole HTTP/1.1" 200 OK
Model loaded: 200


---
## Cell 6: Start Cloudflare Tunnel (gives public URL)

In [54]:
# This will print a URL like https://xxxx.trycloudflare.com
# Keep this cell running to keep the tunnel alive.
!cloudflared tunnel --url http://localhost:8000

2026-05-21T08:54:20Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-21T08:54:20Z INF Requesting new quick Tunnel on trycloudflare.com...
^C


---
## Cell 7: Keep-alive (non-blocking)

In [53]:
from IPython.display import display, Javascript
display(Javascript("""
if (!window._colabKeepalive) {
  window._colabKeepalive = setInterval(function(){}, 60000);
  console.log('Keepalive started.');
}
"""))
print("Keepalive active (browser JS, does not block kernel).")

<IPython.core.display.Javascript object>

Keepalive active (browser JS, does not block kernel).
